# 05 · Tune Temporal Ratio (era : year internal weight)

Finds the optimal internal weight ratio between the era one-hot sub-columns and the
continuous year sub-column in `album_temporal_matrix.npz`.

**What is being tuned:**  
Within the temporal block the era columns always have weight **1.0**. This notebook
sweeps `YEAR_RATIO` (year column weight) from 0.0 (era only) to 1.5 (year equals era)
and evaluates HR@10 / MRR@10 via the fast dot-product method — no KNN rebuild needed.

The Era dial in the app multiplies the **whole** temporal block uniformly on top of
whatever internal ratio is baked in. So tuning the ratio here sets the intra-decade
nuance; the user's dial still controls how much the whole temporal block matters.

**Evaluation:** Last.fm album similarity ground truth, scoped to albums in the
weighted-app matrix universe. Metrics: Hit Rate @10, Precision @10, MRR @10.

**Output:** `data/temporal_best_ratio.json`  
After tuning: rebuild `album_temporal_matrix.npz` in `3-features/14-feature-temporal.ipynb`
with the new YEAR_WEIGHT.

---

Two-phase approach:
- **Phase 1**: 12-point linear sweep over [0.0, 1.5] (~10 min depending on machine)
- **Phase 2**: 10-point fine grid ±40% in linear space around Phase 1 best

In [ ]:
import pickle
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz, hstack, csr_matrix

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
N_RESULTS    = 10
CHUNK_SIZE   = 50_000
K_TOP        = 30

# ── App default dial weights ──────────────────────────────────────────────
# dial_to_weight mirrors the app formula: d/11 * 2.0
def dial_to_weight(d):
    return d / 11 * 2.0

# Weights for the fixed blocks (held at app defaults throughout the sweep)
W_GENRE    = dial_to_weight(6)   # ≈ 1.091
W_LABEL    = dial_to_weight(6)   # ≈ 1.091
W_STATS    = dial_to_weight(6)   # ≈ 1.091
W_COUNTRY  = dial_to_weight(2)   # ≈ 0.364
W_RATINGS  = dial_to_weight(6)   # ≈ 1.091
# Overall temporal block weight — Era dial default is 4
W_TEMPORAL = dial_to_weight(4)   # ≈ 0.727

FIXED_WEIGHTS = {
    'genre':        W_GENRE,
    'record_label': W_LABEL,
    'track_stats':  W_STATS,
    'country':      W_COUNTRY,
    'ratings':      W_RATINGS,
}
print('App default block weights:')
for k, w in FIXED_WEIGHTS.items():
    print(f'  {k:<14}: {w:.4f}')
print(f'  temporal (era dial=4): {W_TEMPORAL:.4f}')

In [ ]:
# ── Load Last.fm ground truth ─────────────────────────────────────────────
# Tries the most likely file paths in order.
# The similarity dataset has columns: album, artist, similar_album, similar_artist
# (one row per seed→similar pair).
_lastfm_paths = [
    f'{DATA_DIR}/lastfm_album_similarity.parquet',
    f'{DATA_DIR}/lastfm_similar_albums.parquet',
    f'{DATA_DIR}/lastfm_data.parquet',
]
lastfm = None
for _p in _lastfm_paths:
    try:
        lastfm = pd.read_parquet(_p)
        print(f'Loaded: {_p}  ({len(lastfm):,} rows)')
        break
    except Exception:
        continue

if lastfm is None:
    raise FileNotFoundError(
        'Could not find Last.fm similarity parquet. Expected columns: '
        'album, artist, similar_album, similar_artist. '
        'Check DATA_DIR or produce the file from 06-evaluate-lastfm.ipynb.'
    )

print(f'Columns: {list(lastfm.columns)}')

In [ ]:
# ── Match Last.fm → MusicBrainz IDs ──────────────────────────────────────
lastfm.columns = lastfm.columns.str.strip()
for col in ['album', 'artist', 'similar_album', 'similar_artist']:
    lastfm[col + '_key'] = lastfm[col].str.lower().str.strip()

mb_lookup = (
    pd.read_parquet(
        f'{DATA_DIR}/mb_album_artists.parquet',
        columns=['album_id', 'album_name', 'artist_name'],
    )
    .drop_duplicates(subset='album_id')
)
mb_lookup['album_key']  = mb_lookup['album_name'].str.lower().str.strip()
mb_lookup['artist_key'] = mb_lookup['artist_name'].str.lower().str.strip()
mb_index = mb_lookup.set_index(['album_key', 'artist_key'])['album_id']

seed_pairs = lastfm[['album_key', 'artist_key']].drop_duplicates()
seed_pairs = seed_pairs.join(
    mb_index.rename('seed_mb_id'), on=['album_key', 'artist_key'], how='left'
)

lastfm = lastfm.join(
    mb_index.rename('similar_mb_id'),
    on=['similar_album_key', 'similar_artist_key'],
    how='left',
)
lastfm = lastfm.merge(seed_pairs, on=['album_key', 'artist_key'], how='left')

ground_truth_raw = (
    lastfm
    .dropna(subset=['seed_mb_id', 'similar_mb_id'])
    .groupby('seed_mb_id')['similar_mb_id']
    .apply(set)
    .to_dict()
)
ground_truth_raw = {
    int(k): {int(v) for v in vs} for k, vs in ground_truth_raw.items()
}
print(f'Ground truth: {len(ground_truth_raw):,} seed albums')

In [ ]:
# ── Load feature blocks ───────────────────────────────────────────────────
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

print('Loading feature blocks...')
X_genre    = load_npz(f'{FEATURES_DIR}/album_genre_matrix.npz')
X_label    = load_npz(f'{FEATURES_DIR}/album_record_label_matrix.npz')
X_stats    = load_npz(f'{FEATURES_DIR}/album_track_stats_matrix.npz')
X_country  = load_npz(f'{FEATURES_DIR}/album_country_matrix.npz')
X_ratings  = load_npz(f'{FEATURES_DIR}/album_ratings_matrix.npz')
X_era      = load_npz(f'{FEATURES_DIR}/album_era_matrix.npz')
X_year_col = load_npz(f'{FEATURES_DIR}/album_year_col.npz')

BLOCKS_ALL = [X_genre, X_label, X_stats, X_country, X_ratings, X_era, X_year_col]
n_albums   = len(album_id_order)
for name, X in zip(['genre', 'record_label', 'track_stats', 'country',
                    'ratings', 'era', 'year_col'], BLOCKS_ALL):
    print(f'  {name:<14}: {X.shape}  nnz={X.nnz:,}')

In [ ]:
# ── Build annotated album index ───────────────────────────────────────────
# An album is 'annotated' if it has any non-zero entry across the combined matrix.
# This mirrors the app: albums with no features in any block are never returned.
X_all_check = hstack(BLOCKS_ALL, format='csr')
has_feat    = np.asarray(X_all_check.sum(axis=1)).ravel() > 0

ann_ids    = np.array(album_id_order)[has_feat]
id2row_ann = {int(aid): i for i, aid in enumerate(ann_ids)}
n_ann      = int(has_feat.sum())
all_model_ids = set(int(x) for x in ann_ids)

print(f'Annotated albums: {n_ann:,} ({n_ann / n_albums * 100:.1f}%)')

def _clean(X):
    """Subset to annotated rows, ensure float32, remove infinities."""
    M = X[has_feat].tocsr().astype(np.float32)
    np.nan_to_num(M.data, nan=0.0, posinf=0.0, neginf=0.0, copy=False)
    M.eliminate_zeros()
    return M

BLK = {
    'genre':        _clean(X_genre),
    'record_label': _clean(X_label),
    'track_stats':  _clean(X_stats),
    'country':      _clean(X_country),
    'ratings':      _clean(X_ratings),
    'era':          _clean(X_era),
    'year_col':     _clean(X_year_col),
}
BLK_ROW_SQ = {
    k: np.asarray(M.power(2).sum(axis=1)).ravel().astype(np.float32)
    for k, M in BLK.items()
}

# Scope-filter ground truth: both seed and similar must be in annotated index
ground_truth = {
    k: {v for v in vs if v in all_model_ids}
    for k, vs in ground_truth_raw.items()
}
ground_truth = {
    k: vs for k, vs in ground_truth.items()
    if k in id2row_ann and vs
}

seed_ids   = list(ground_truth.keys())
seed_rows  = np.array([id2row_ann[s] for s in seed_ids], dtype=np.int32)
seed_known = [ground_truth[s] for s in seed_ids]
n_seeds    = len(seed_ids)
print(f'Scoped seeds for eval: {n_seeds:,}')

In [ ]:
# ── Pre-compute seed dot products ─────────────────────────────────────────
# ERA_DOTS and YEAR_DOTS are stored separately so we can sweep year_ratio
# without recomputing anything. FIXED_DOTS folds in all non-temporal blocks.
#
# Memory estimate:
#   ERA_DOTS  (n_seeds × n_ann) float16  ≈ n_seeds × n_ann × 2 bytes
#   YEAR_DOTS (n_seeds × n_ann) float16  ≈ same
#   FIXED_DOTS float32                   ≈ double
# For n_seeds≈2000, n_ann≈1.76M: each float16 slab ≈ 7 GB. If OOM, reduce
# CHUNK_SIZE in the eval loop — it won't need the pre-computed slabs.

print('Pre-computing ERA_DOTS ...')
t0 = time.time()
M_era  = BLK['era']
Q_era  = M_era[seed_rows]
ERA_DOTS          = np.ascontiguousarray(M_era.dot(Q_era.T).toarray().T.astype(np.float16))
ERA_SEED_NORMS_SQ = BLK_ROW_SQ['era'][seed_rows].astype(np.float32)
ERA_CAND_SQ       = BLK_ROW_SQ['era'].astype(np.float32)
print(f'  ERA_DOTS: {ERA_DOTS.shape}  [{time.time()-t0:.1f}s]')

print('Pre-computing YEAR_DOTS ...')
t0 = time.time()
M_year  = BLK['year_col']
Q_year  = M_year[seed_rows]
YEAR_DOTS          = np.ascontiguousarray(M_year.dot(Q_year.T).toarray().T.astype(np.float16))
YEAR_SEED_NORMS_SQ = BLK_ROW_SQ['year_col'][seed_rows].astype(np.float32)
YEAR_CAND_SQ       = BLK_ROW_SQ['year_col'].astype(np.float32)
print(f'  YEAR_DOTS: {YEAR_DOTS.shape}  [{time.time()-t0:.1f}s]')

print('Pre-computing FIXED_DOTS ...')
t0 = time.time()
FIXED_DOTS     = np.zeros((n_seeds, n_ann), dtype=np.float32)
FIXED_NORMS_SQ = np.zeros(n_seeds,         dtype=np.float32)
FIXED_CAND_SQ  = np.zeros(n_ann,           dtype=np.float32)

for blk_name, w in FIXED_WEIGHTS.items():
    if w == 0.0:
        continue
    w2 = np.float32(w ** 2)
    M  = BLK[blk_name]
    Q  = M[seed_rows]
    D  = M.dot(Q.T).toarray().T.astype(np.float32)
    FIXED_DOTS     += w2 * D
    FIXED_NORMS_SQ += w2 * BLK_ROW_SQ[blk_name][seed_rows]
    FIXED_CAND_SQ  += w2 * BLK_ROW_SQ[blk_name]
    print(f'  added {blk_name:<14} w={w:.4f}')

print(f'FIXED_DOTS done  [{time.time()-t0:.1f}s]')
print()
print('All pre-computations complete. Eval loop will be fast.')

In [ ]:
# ── Fast evaluation function ──────────────────────────────────────────────
# Computes HR@N, Precision@N, MRR@N for a given year_ratio.
# ERA_WEIGHT is always 1.0; year column weight = W_TEMPORAL * year_ratio.

def evaluate_ratio(year_ratio: float, n: int = N_RESULTS) -> dict:
    # Effective squared weights inside the temporal block
    era_w2  = np.float32((W_TEMPORAL * 1.0)        ** 2)
    year_w2 = np.float32((W_TEMPORAL * year_ratio) ** 2)

    # ── Norms ─────────────────────────────────────────────────────────────
    cand_norm_sq = FIXED_CAND_SQ.copy()
    cand_norm_sq += era_w2  * ERA_CAND_SQ
    cand_norm_sq += year_w2 * YEAR_CAND_SQ

    q_norm_sq = FIXED_NORMS_SQ.copy()
    q_norm_sq += era_w2  * ERA_SEED_NORMS_SQ
    q_norm_sq += year_w2 * YEAR_SEED_NORMS_SQ

    cand_norms = np.sqrt(cand_norm_sq); cand_norms[cand_norms == 0] = 1.0
    q_norms    = np.sqrt(q_norm_sq);    q_norms[q_norms == 0]       = 1.0

    # ── Chunked cosine similarity ──────────────────────────────────────────
    best_scores = np.full((n_seeds, K_TOP), -np.inf, dtype=np.float32)
    best_cols   = np.zeros((n_seeds, K_TOP),          dtype=np.int32)
    chunk_buf   = np.empty((n_seeds, CHUNK_SIZE),     dtype=np.float32)

    for col_start in range(0, n_ann, CHUNK_SIZE):
        col_end = min(col_start + CHUNK_SIZE, n_ann)
        csize   = col_end - col_start
        chunk   = chunk_buf[:, :csize]

        chunk[:] = FIXED_DOTS[:, col_start:col_end]
        chunk   += era_w2  * ERA_DOTS[ :, col_start:col_end].astype(np.float32)
        chunk   += year_w2 * YEAR_DOTS[:, col_start:col_end].astype(np.float32)

        # Cosine normalise
        chunk /= q_norms[:, None]
        chunk /= cand_norms[None, col_start:col_end]

        # Zero out self-loops
        self_mask = (seed_rows >= col_start) & (seed_rows < col_end)
        for s_idx in np.where(self_mask)[0]:
            chunk[s_idx, seed_rows[s_idx] - col_start] = -1.0

        # Top-K within chunk
        k_use = min(K_TOP, csize)
        chunk_top_idx    = np.argpartition(chunk, -k_use, axis=1)[:, -k_use:]
        chunk_top_scores = chunk[np.arange(n_seeds)[:, None], chunk_top_idx]
        chunk_top_cols   = (chunk_top_idx + col_start).astype(np.int32)

        combined_scores = np.concatenate([best_scores, chunk_top_scores], axis=1)
        combined_cols   = np.concatenate([best_cols,   chunk_top_cols],   axis=1)
        keep            = np.argpartition(combined_scores, -K_TOP, axis=1)[:, -K_TOP:]
        best_scores     = combined_scores[np.arange(n_seeds)[:, None], keep]
        best_cols       = combined_cols[  np.arange(n_seeds)[:, None], keep]

    # ── Metrics ───────────────────────────────────────────────────────────
    order      = np.argsort(best_scores, axis=1)[:, ::-1]
    sorted_idx = best_cols[np.arange(n_seeds)[:, None], order]

    hits, precs, rrs = [], [], []
    for s_idx in range(n_seeds):
        known = seed_known[s_idx]
        s_id  = seed_ids[s_idx]
        recs  = [int(ann_ids[i]) for i in sorted_idx[s_idx]
                 if int(ann_ids[i]) != s_id][:n]
        overlap = set(recs) & known
        hits.append(1 if overlap else 0)
        precs.append(len(overlap) / n)
        rr = 0.0
        for rank, aid in enumerate(recs, 1):
            if aid in known:
                rr = 1.0 / rank
                break
        rrs.append(rr)

    return {
        'hit_rate':  float(np.mean(hits)),
        'precision': float(np.mean(precs)),
        'mrr':       float(np.mean(rrs)),
        'n_eval':    len(hits),
    }

In [ ]:
# ── Phase 1: coarse sweep ─────────────────────────────────────────────────
# Includes 0.0 (era only) as baseline. Covers the likely useful range [0, 1.5].
PHASE1_RATIOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.25, 1.5]

p1_results = []
print(f'Phase 1: {len(PHASE1_RATIOS)} points')
print(f'{"year_ratio":>12}  {"hit_rate":>10}  {"precision":>10}  {"mrr":>10}  {"secs":>6}')
print('-' * 60)

for year_ratio in PHASE1_RATIOS:
    t0     = time.time()
    scores = evaluate_ratio(year_ratio)
    elapsed = time.time() - t0
    scores['year_ratio'] = year_ratio
    p1_results.append(scores)
    print(
        f'{year_ratio:>12.3f}  '
        f'{scores["hit_rate"]:>10.4f}  '
        f'{scores["precision"]:>10.4f}  '
        f'{scores["mrr"]:>10.4f}  '
        f'{elapsed:>6.1f}s'
    )

p1_df   = pd.DataFrame(p1_results)
p1_best = p1_df.loc[p1_df['hit_rate'].idxmax()]
print(f'\nPhase 1 best  year_ratio={p1_best["year_ratio"]:.3f}  hit_rate={p1_best["hit_rate"]:.4f}')

In [ ]:
# ── Phase 1 plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['hit_rate', 'precision', 'mrr']):
    ax.plot(p1_df['year_ratio'], p1_df[metric], 'o-', linewidth=2)
    ax.axvline(p1_best['year_ratio'], color='red', linestyle='--',
               label=f'best={p1_best["year_ratio"]:.2f}')
    ax.axvline(0.0, color='grey', linestyle=':', linewidth=1,
               label='era only')
    ax.set_xlabel('year_ratio (era=1.0, year=ratio)')
    ax.set_ylabel(metric.replace('_', ' ').title() + ' @10')
    ax.set_title(metric.replace('_', ' ').title())
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Phase 1: coarse sweep — era:year internal ratio', y=1.02)
plt.tight_layout()
plt.savefig('tune_temporal_phase1.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved tune_temporal_phase1.png')

In [ ]:
# ── Phase 2: fine grid around Phase 1 best ────────────────────────────────
best_ratio = float(p1_best['year_ratio'])

# If the best is at 0.0 (era only wins), run a narrow low-ratio grid
if best_ratio == 0.0:
    p2_lo, p2_hi = 0.0, 0.25
else:
    p2_lo = max(0.0, best_ratio * 0.6)
    p2_hi = best_ratio * 1.4

PHASE2_RATIOS = np.linspace(p2_lo, p2_hi, 10).tolist()

p2_results = []
print(f'Phase 2: 10 points in [{p2_lo:.3f}, {p2_hi:.3f}]')
print(f'{"year_ratio":>12}  {"hit_rate":>10}  {"precision":>10}  {"mrr":>10}  {"secs":>6}')
print('-' * 60)

for year_ratio in PHASE2_RATIOS:
    t0     = time.time()
    scores = evaluate_ratio(year_ratio)
    elapsed = time.time() - t0
    scores['year_ratio'] = year_ratio
    p2_results.append(scores)
    print(
        f'{year_ratio:>12.3f}  '
        f'{scores["hit_rate"]:>10.4f}  '
        f'{scores["precision"]:>10.4f}  '
        f'{scores["mrr"]:>10.4f}  '
        f'{elapsed:>6.1f}s'
    )

p2_df   = pd.DataFrame(p2_results)
p2_best = p2_df.loc[p2_df['hit_rate'].idxmax()]
print(f'\nPhase 2 best  year_ratio={p2_best["year_ratio"]:.3f}  hit_rate={p2_best["hit_rate"]:.4f}')

In [ ]:
# ── Final plot: Phase 1 + Phase 2 combined ────────────────────────────────
all_results = pd.concat([p1_df, p2_df], ignore_index=True).sort_values('year_ratio')
best_final  = all_results.loc[all_results['hit_rate'].idxmax()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['hit_rate', 'precision', 'mrr']):
    ax.scatter(p1_df['year_ratio'], p1_df[metric],
               alpha=0.6, label='Phase 1', color='steelblue')
    ax.scatter(p2_df['year_ratio'], p2_df[metric],
               alpha=0.9, label='Phase 2', color='darkorange', marker='D', s=60)
    ax.axvline(best_final['year_ratio'], color='red', linestyle='--',
               label=f'optimal={best_final["year_ratio"]:.3f}')
    ax.axvline(0.0, color='grey', linestyle=':', linewidth=1,
               label='era only baseline')
    ax.set_xlabel('year_ratio')
    ax.set_ylabel(metric.replace('_', ' ').title() + ' @10')
    ax.set_title(metric.replace('_', ' ').title())
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Temporal ratio tuning — era:year internal weight', y=1.02)
plt.tight_layout()
plt.savefig('tune_temporal_final.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved tune_temporal_final.png')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
era_only = all_results.loc[all_results['year_ratio'].sub(0.0).abs().idxmin()]

print('=' * 60)
print('RESULTS')
print('=' * 60)
print(f'  Era only  (year_ratio=0.0)   hit={era_only["hit_rate"]:.4f}  '
      f'prec={era_only["precision"]:.4f}  mrr={era_only["mrr"]:.4f}')
print(f'  Optimal   (year_ratio={best_final["year_ratio"]:.3f})  '
      f'hit={best_final["hit_rate"]:.4f}  '
      f'prec={best_final["precision"]:.4f}  '
      f'mrr={best_final["mrr"]:.4f}')
print()
delta = best_final['hit_rate'] - era_only['hit_rate']
print(f'  Δ hit_rate = {delta:+.4f}  '
      f'({delta / era_only["hit_rate"] * 100:+.1f}% vs era only)'
      if era_only['hit_rate'] > 0 else f'  Δ hit_rate = {delta:+.4f}')
print()
if delta <= 0:
    print('  ⚠  Year signal does not improve HR@10.')
    print('  Recommendation: keep YEAR_WEIGHT=0.0 (era only).')
    print('  If this is consistent across MRR and Precision, Option A is correct.')
else:
    print(f'  ✓  Optimal YEAR_WEIGHT = {best_final["year_ratio"]:.3f}')
    print(f'  Rebuild album_temporal_matrix.npz with YEAR_WEIGHT={best_final["year_ratio"]:.3f}')

In [ ]:
# ── Save result ───────────────────────────────────────────────────────────
YEAR_RATIO_FINAL = float(best_final['year_ratio'])

result = {
    'year_ratio':          YEAR_RATIO_FINAL,
    'era_weight_internal': 1.0,
    'year_weight_internal': YEAR_RATIO_FINAL,
    'era_dial_default':    4,
    'W_TEMPORAL':          W_TEMPORAL,
    'metrics_era_only': {
        'hit_rate':  float(era_only['hit_rate']),
        'precision': float(era_only['precision']),
        'mrr':       float(era_only['mrr']),
    },
    'metrics_optimal': {
        'hit_rate':  float(best_final['hit_rate']),
        'precision': float(best_final['precision']),
        'mrr':       float(best_final['mrr']),
    },
    'n_seeds_evaluated': int(best_final['n_eval']),
}

with open(f'{DATA_DIR}/temporal_best_ratio.json', 'w') as f:
    json.dump(result, f, indent=2)

print('Saved temporal_best_ratio.json')
print()
print('Next steps:')
print(f'  1. Open 3-features/14-feature-temporal.ipynb')
print(f'     Set YEAR_WEIGHT = {YEAR_RATIO_FINAL:.3f}')
print(f'     Re-run to rebuild album_temporal_matrix.npz')
print(f'  2. In 5-app/app_v3_weighted.py swap:')
print(f"     'era': 'album_era_matrix.npz'")
print(f"     → 'era': 'album_temporal_matrix.npz'")